In [1]:
# ============================================================
# PHASE 25 — FINAL MANUSCRIPT RESULTS INTEGRATION PACKAGE
# ============================================================
# Goal:
# Integrate outputs from Phases 17–24 into manuscript-ready tables,
# wording, figure recommendations, and claim-strength audit.
#
# This phase does not train new models.
# It creates:
# 1. Final result summary table
# 2. Final claim-strength table
# 3. Main-text figure recommendation
# 4. Supplementary material recommendation
# 5. Manuscript-ready Results/Discussion wording
# 6. Reviewer-response-ready limitations
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_colwidth", 500)

import os
from pathlib import Path
import pandas as pd
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive đã được kết nối từ trước.")

PROJECT_DIR = Path("/content/drive/MyDrive/Project_Protein")

PHASE17_DIR = PROJECT_DIR / "model" / "phase17_repeated_cross_validation_robust_benchmark"
PHASE18_DIR = PROJECT_DIR / "model" / "phase18_negative_set_sensitivity_analysis"
PHASE19_DIR = PROJECT_DIR / "model" / "phase19_external_independent_gene_set_validation"
PHASE19B_DIR = PROJECT_DIR / "model" / "phase19b_external_supported_negative_audit"
PHASE20_DIR = PROJECT_DIR / "model" / "phase20_pathway_network_validation"
PHASE21_DIR = PROJECT_DIR / "model" / "phase21_case_level_explainability_error_ranking_audit"
PHASE22_DIR = PROJECT_DIR / "model" / "phase22_multimodal_fusion_strategy_comparison"
PHASE23_DIR = PROJECT_DIR / "model" / "phase23_leakage_and_bias_audit"
PHASE23B_DIR = PROJECT_DIR / "model" / "phase23b_bias_controlled_robustness_audit"
PHASE24_DIR = PROJECT_DIR / "model" / "phase24_topk_ranking_utility_metrics"

PHASE25_DIR = PROJECT_DIR / "model" / "phase25_final_manuscript_results_integration"
RESULT_DIR = PHASE25_DIR / "results"
EXCEL_DIR = PHASE25_DIR / "excel"
REPORT_DIR = PHASE25_DIR / "reports"

for d in [PHASE25_DIR, RESULT_DIR, EXCEL_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Phase 25 output:", PHASE25_DIR)

Mounted at /content/drive
Phase 25 output: /content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration


In [2]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def safe_read_csv(path):
    path = Path(path)
    if path.exists():
        print("Loaded:", path)
        return pd.read_csv(path)
    print("Missing:", path)
    return pd.DataFrame()


def save_df(df, path):
    path = Path(path)
    df.to_csv(path, index=False)
    print("Saved:", path)


def round_numeric_columns(df, digits=4):
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            df[col] = df[col].round(digits)
    return df


def write_markdown_section(f, title, text):
    f.write(f"## {title}\n\n")
    f.write(text.strip() + "\n\n")

In [3]:
# ============================================================
# LOAD IMPORTANT OUTPUTS
# ============================================================

phase17_cv = safe_read_csv(
    PHASE17_DIR / "results" / "phase17_manuscript_repeated_cv_table.csv"
)

phase17_paired = safe_read_csv(
    PHASE17_DIR / "results" / "phase17_repeated_cv_paired_statistical_tests.csv"
)

phase18_summary = safe_read_csv(
    PHASE18_DIR / "results" / "phase18_negative_sensitivity_summary_wide.csv"
)

phase22_summary = safe_read_csv(
    PHASE22_DIR / "results" / "phase22_fusion_strategy_summary_wide.csv"
)

phase22_paired = safe_read_csv(
    PHASE22_DIR / "results" / "phase22_paired_fusion_strategy_tests.csv"
)

phase22_weights = safe_read_csv(
    PHASE22_DIR / "results" / "phase22_late_fusion_weight_summary.csv"
)

phase23_duplicate = safe_read_csv(
    PHASE23_DIR / "results" / "phase23_duplicate_audit_summary.csv"
)

phase23_nn = safe_read_csv(
    PHASE23_DIR / "results" / "phase23_original_split_nearest_neighbor_summary.csv"
)

phase23_nuisance = safe_read_csv(
    PHASE23_DIR / "results" / "phase23_nuisance_only_cv_summary.csv"
)

phase23_cluster = safe_read_csv(
    PHASE23_DIR / "results" / "phase23_cluster_group_cv_degradation_summary.csv"
)

phase23b_nuisance = safe_read_csv(
    PHASE23B_DIR / "results" / "phase23b_nuisance_feature_set_summary_wide.csv"
)

phase23b_matched = safe_read_csv(
    PHASE23B_DIR / "results" / "phase23b_matched_vs_original_repeated_cv_comparison.csv"
)

phase24_topk = safe_read_csv(
    PHASE24_DIR / "results" / "phase24_manuscript_topk_current_label_utility_table.csv"
)

phase24_random = safe_read_csv(
    PHASE24_DIR / "results" / "phase24_manuscript_topk_random_baseline_table.csv"
)

phase24_bio = safe_read_csv(
    PHASE24_DIR / "results" / "phase24_manuscript_topk_biological_support_table.csv"
)

phase20_pathway = safe_read_csv(
    PHASE20_DIR / "results" / "phase20_manuscript_top100_pathway_validation_table.csv"
)

phase20_network = safe_read_csv(
    PHASE20_DIR / "results" / "phase20_manuscript_network_density_table.csv"
)

phase21_cases = safe_read_csv(
    PHASE21_DIR / "results" / "phase21_manuscript_gene_case_table.csv"
)

Loaded: /content/drive/MyDrive/Project_Protein/model/phase17_repeated_cross_validation_robust_benchmark/results/phase17_manuscript_repeated_cv_table.csv
Loaded: /content/drive/MyDrive/Project_Protein/model/phase17_repeated_cross_validation_robust_benchmark/results/phase17_repeated_cv_paired_statistical_tests.csv
Loaded: /content/drive/MyDrive/Project_Protein/model/phase18_negative_set_sensitivity_analysis/results/phase18_negative_sensitivity_summary_wide.csv
Loaded: /content/drive/MyDrive/Project_Protein/model/phase22_multimodal_fusion_strategy_comparison/results/phase22_fusion_strategy_summary_wide.csv
Loaded: /content/drive/MyDrive/Project_Protein/model/phase22_multimodal_fusion_strategy_comparison/results/phase22_paired_fusion_strategy_tests.csv
Loaded: /content/drive/MyDrive/Project_Protein/model/phase22_multimodal_fusion_strategy_comparison/results/phase22_late_fusion_weight_summary.csv
Loaded: /content/drive/MyDrive/Project_Protein/model/phase23_leakage_and_bias_audit/results/pha

In [4]:
# ============================================================
# FINAL MODEL PERFORMANCE SUMMARY
# ============================================================

final_performance_records = []

if not phase17_cv.empty:
    tmp = phase17_cv.copy()

    for _, row in tmp.iterrows():
        final_performance_records.append({
            "evidence_layer": "Repeated CV benchmark",
            "model_or_analysis": row.get("display_name", row.get("model_name")),
            "main_result": (
                f"PR-AUC {row.get('pr_auc_mean', np.nan):.3f} ± {row.get('pr_auc_sd', np.nan):.3f}; "
                f"MCC {row.get('mcc_mean', np.nan):.3f} ± {row.get('mcc_sd', np.nan):.3f}; "
                f"ROC-AUC {row.get('roc_auc_mean', np.nan):.3f} ± {row.get('roc_auc_sd', np.nan):.3f}"
            ),
            "interpretation": "Core internal robustness benchmark."
        })

if not phase22_summary.empty:
    best_pr = phase22_summary.sort_values("pr_auc_mean", ascending=False).iloc[0]
    best_mcc = phase22_summary.sort_values("mcc_mean", ascending=False).iloc[0]

    final_performance_records.append({
        "evidence_layer": "Fusion strategy comparison",
        "model_or_analysis": "Best PR-AUC fusion",
        "main_result": f"{best_pr['fusion_model']} achieved PR-AUC {best_pr['pr_auc_mean']:.3f} ± {best_pr['pr_auc_sd']:.3f}",
        "interpretation": "Decision-level fusion provides small ranking gains."
    })

    final_performance_records.append({
        "evidence_layer": "Fusion strategy comparison",
        "model_or_analysis": "Best MCC fusion",
        "main_result": f"{best_mcc['fusion_model']} achieved MCC {best_mcc['mcc_mean']:.3f} ± {best_mcc['mcc_sd']:.3f}",
        "interpretation": "Early DNABERT-2 fusion remains strong for threshold-based classification."
    })

if not phase24_topk.empty:
    top100 = phase24_topk[phase24_topk["top_k"] == 100].sort_values("precision_at_k", ascending=False)
    if not top100.empty:
        row = top100.iloc[0]
        final_performance_records.append({
            "evidence_layer": "Top-K ranking utility",
            "model_or_analysis": row["display_name"],
            "main_result": f"Top-100 recovered {int(row['n_positive_in_topk'])}/100 positives; Precision@100={row['precision_at_k']:.3f}",
            "interpretation": "Strong gene-prioritization utility."
        })

final_performance_summary_df = pd.DataFrame(final_performance_records)
display(final_performance_summary_df)

save_df(
    final_performance_summary_df,
    RESULT_DIR / "phase25_final_performance_summary.csv"
)

,evidence_layer,model_or_analysis,main_result,interpretation
0,Repeated CV benchmark,DNABERT-2 multimodal,PR-AUC 0.716 ± 0.024; MCC 0.331 ± 0.052; ROC-AUC 0.731 ± 0.023,Core internal robustness benchmark.
1,Repeated CV benchmark,Handcrafted multimodal,PR-AUC 0.714 ± 0.019; MCC 0.317 ± 0.049; ROC-AUC 0.725 ± 0.020,Core internal robustness benchmark.
2,Repeated CV benchmark,Protein-only,PR-AUC 0.695 ± 0.024; MCC 0.308 ± 0.033; ROC-AUC 0.713 ± 0.020,Core internal robustness benchmark.
3,Repeated CV benchmark,Genomic-only,PR-AUC 0.645 ± 0.029; MCC 0.203 ± 0.043; ROC-AUC 0.641 ± 0.029,Core internal robustness benchmark.
4,Fusion strategy comparison,Best PR-AUC fusion,Stacking_Protein_Handcrafted_DNABERT2 achieved PR-AUC 0.721 ± 0.020,Decision-level fusion provides small ranking gains.
5,Fusion strategy comparison,Best MCC fusion,Early_fusion_Protein_DNABERT2Genomic achieved MCC 0.339 ± 0.054,Early DNABERT-2 fusion remains strong for threshold-based classification.
6,Top-K ranking utility,Handcrafted multimodal,Top-100 recovered 84/100 positives; Precision@100=0.840,Strong gene-prioritization utility.


Saved: /content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/results/phase25_final_performance_summary.csv


In [5]:
# ============================================================
# CLAIM STRENGTH AUDIT
# ============================================================

claim_records = [
    {
        "claim": "Protein-containing models outperform genomic-only models.",
        "support_level": "Strong",
        "evidence": "Repeated CV, negative sensitivity, protein-only vs genomic-only paired results.",
        "safe_wording": "Protein-containing models consistently outperformed the genomic-only baseline."
    },
    {
        "claim": "Multimodal models improve ranking over protein-only.",
        "support_level": "Moderate to strong",
        "evidence": "Repeated CV PR-AUC improvements; Top-K utility; Phase 22 fusion comparison.",
        "safe_wording": "Multimodal models improved ranking-oriented metrics relative to protein-only models, although gains were modest."
    },
    {
        "claim": "DNABERT-2 is superior to handcrafted multimodal.",
        "support_level": "Weak / not established",
        "evidence": "DNABERT-2 had small directional gains in repeated CV and some fusion settings, but not consistently significant after correction.",
        "safe_wording": "DNABERT-2 provided a directional ranking extension but did not establish clear superiority over handcrafted genomic features."
    },
    {
        "claim": "Top-K gene prioritization utility is strong.",
        "support_level": "Strong",
        "evidence": "Top-K current-positive recovery substantially exceeded random baselines across K thresholds.",
        "safe_wording": "Top-K analysis demonstrated strong prioritization utility, with multimodal rankings recovering substantially more current-positive genes than random same-size gene sets."
    },
    {
        "claim": "External/pathway biological validation is confirmatory.",
        "support_level": "Weak",
        "evidence": "External/pathway enrichment did not remain significant after FDR correction in Phases 19–20.",
        "safe_wording": "Biological validation was supportive and hypothesis-generating rather than confirmatory."
    },
    {
        "claim": "Mitochondrial/OXPHOS module is biologically meaningful.",
        "support_level": "Supportive / hypothesis-generating",
        "evidence": "Case-level and network analyses highlighted NDUF/OXPHOS candidates, but FDR-corrected enrichment was not significant.",
        "safe_wording": "Mitochondrial/OXPHOS candidates emerged as a hypothesis-generating module-level signal."
    },
    {
        "claim": "The study is free from leakage and bias.",
        "support_level": "Not acceptable",
        "evidence": "No serious duplicate leakage, but nuisance-only signal and cluster-aware performance drop exist.",
        "safe_wording": "Leakage and bias audits did not reveal severe duplication leakage, but residual dataset-level bias and embedding-neighborhood effects were detected and explicitly controlled."
    },
    {
        "claim": "Bias controls preserve main conclusions.",
        "support_level": "Moderate to strong",
        "evidence": "Gene-symbol-length matching did not reduce main model performance.",
        "safe_wording": "Bias-controlled matching showed that gene-symbol-length imbalance did not explain the main sequence-based ranking results."
    }
]

claim_strength_df = pd.DataFrame(claim_records)
display(claim_strength_df)

save_df(
    claim_strength_df,
    RESULT_DIR / "phase25_claim_strength_audit.csv"
)

,claim,support_level,evidence,safe_wording
0,Protein-containing models outperform genomic-only models.,Strong,"Repeated CV, negative sensitivity, protein-only vs genomic-only paired results.",Protein-containing models consistently outperformed the genomic-only baseline.
1,Multimodal models improve ranking over protein-only.,Moderate to strong,Repeated CV PR-AUC improvements; Top-K utility; Phase 22 fusion comparison.,"Multimodal models improved ranking-oriented metrics relative to protein-only models, although gains were modest."
2,DNABERT-2 is superior to handcrafted multimodal.,Weak / not established,"DNABERT-2 had small directional gains in repeated CV and some fusion settings, but not consistently significant after correction.",DNABERT-2 provided a directional ranking extension but did not establish clear superiority over handcrafted genomic features.
3,Top-K gene prioritization utility is strong.,Strong,Top-K current-positive recovery substantially exceeded random baselines across K thresholds.,"Top-K analysis demonstrated strong prioritization utility, with multimodal rankings recovering substantially more current-positive genes than random same-size gene sets."
4,External/pathway biological validation is confirmatory.,Weak,External/pathway enrichment did not remain significant after FDR correction in Phases 19–20.,Biological validation was supportive and hypothesis-generating rather than confirmatory.
5,Mitochondrial/OXPHOS module is biologically meaningful.,Supportive / hypothesis-generating,"Case-level and network analyses highlighted NDUF/OXPHOS candidates, but FDR-corrected enrichment was not significant.",Mitochondrial/OXPHOS candidates emerged as a hypothesis-generating module-level signal.
6,The study is free from leakage and bias.,Not acceptable,"No serious duplicate leakage, but nuisance-only signal and cluster-aware performance drop exist.","Leakage and bias audits did not reveal severe duplication leakage, but residual dataset-level bias and embedding-neighborhood effects were detected and explicitly controlled."
7,Bias controls preserve main conclusions.,Moderate to strong,Gene-symbol-length matching did not reduce main model performance.,Bias-controlled matching showed that gene-symbol-length imbalance did not explain the main sequence-based ranking results.


Saved: /content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/results/phase25_claim_strength_audit.csv


In [6]:
# ============================================================
# FIGURE RECOMMENDATION TABLE
# ============================================================

figure_records = [
    {
        "figure_id": "Figure 1",
        "recommended_content": "Overall workflow diagram",
        "source_phase": "Phase 16",
        "main_or_supplement": "Main",
        "reason": "Shows dataset construction, feature extraction, evaluation, XAI and validation pipeline."
    },
    {
        "figure_id": "Figure 2",
        "recommended_content": "Repeated CV model performance",
        "source_phase": "Phase 17",
        "main_or_supplement": "Main",
        "reason": "Core benchmark result."
    },
    {
        "figure_id": "Figure 3",
        "recommended_content": "Top-K Precision/Enrichment and observed vs random Top-100 recovery",
        "source_phase": "Phase 24",
        "main_or_supplement": "Main",
        "reason": "Best evidence for gene-prioritization utility."
    },
    {
        "figure_id": "Figure 4",
        "recommended_content": "Fusion strategy comparison",
        "source_phase": "Phase 22",
        "main_or_supplement": "Main or Supplementary",
        "reason": "Supports multimodal integration analysis."
    },
    {
        "figure_id": "Figure 5",
        "recommended_content": "Block-level modality importance",
        "source_phase": "Phase 14 / 16",
        "main_or_supplement": "Main",
        "reason": "Supports explainable modality contribution."
    },
    {
        "figure_id": "Figure 6",
        "recommended_content": "Case-level candidate/pathway-supported genes",
        "source_phase": "Phase 21",
        "main_or_supplement": "Main or Supplementary",
        "reason": "Strengthens explainability under incomplete labels."
    },
    {
        "figure_id": "Supplementary Figure S1",
        "recommended_content": "Negative-set sensitivity plots",
        "source_phase": "Phase 18",
        "main_or_supplement": "Supplementary",
        "reason": "Robustness evidence."
    },
    {
        "figure_id": "Supplementary Figure S2",
        "recommended_content": "Pathway/network validation plots",
        "source_phase": "Phase 20",
        "main_or_supplement": "Supplementary",
        "reason": "Supportive but not confirmatory biological evidence."
    },
    {
        "figure_id": "Supplementary Figure S3",
        "recommended_content": "Leakage/bias audit plots",
        "source_phase": "Phase 23 / 23B",
        "main_or_supplement": "Supplementary",
        "reason": "Addresses reviewer concerns without overcrowding main text."
    }
]

figure_recommendation_df = pd.DataFrame(figure_records)
display(figure_recommendation_df)

save_df(
    figure_recommendation_df,
    RESULT_DIR / "phase25_figure_recommendation_table.csv"
)

,figure_id,recommended_content,source_phase,main_or_supplement,reason
0,Figure 1,Overall workflow diagram,Phase 16,Main,"Shows dataset construction, feature extraction, evaluation, XAI and validation pipeline."
1,Figure 2,Repeated CV model performance,Phase 17,Main,Core benchmark result.
2,Figure 3,Top-K Precision/Enrichment and observed vs random Top-100 recovery,Phase 24,Main,Best evidence for gene-prioritization utility.
3,Figure 4,Fusion strategy comparison,Phase 22,Main or Supplementary,Supports multimodal integration analysis.
4,Figure 5,Block-level modality importance,Phase 14 / 16,Main,Supports explainable modality contribution.
5,Figure 6,Case-level candidate/pathway-supported genes,Phase 21,Main or Supplementary,Strengthens explainability under incomplete labels.
6,Supplementary Figure S1,Negative-set sensitivity plots,Phase 18,Supplementary,Robustness evidence.
7,Supplementary Figure S2,Pathway/network validation plots,Phase 20,Supplementary,Supportive but not confirmatory biological evidence.
8,Supplementary Figure S3,Leakage/bias audit plots,Phase 23 / 23B,Supplementary,Addresses reviewer concerns without overcrowding main text.


Saved: /content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/results/phase25_figure_recommendation_table.csv


In [7]:
# ============================================================
# MANUSCRIPT-READY FINAL WORDING
# ============================================================

abstract_results_text = """
Across repeated stratified cross-validation, protein-containing models consistently outperformed the genomic-only baseline. Multimodal models improved ranking-oriented metrics relative to protein-only models, although gains were modest. DNABERT-2 multimodal representations achieved directional improvements in some settings but did not establish clear superiority over handcrafted genomic features after multiple-testing correction. Top-K ranking analysis demonstrated strong gene-prioritization utility, with multimodal models recovering substantially more current-positive genes in top-ranked lists than expected under random same-size gene baselines. Biological validation supported selected T2D-relevant mechanisms, particularly mitochondrial/OXPHOS and glucose-regulatory candidates, but did not provide confirmatory pathway enrichment after correction.
""".strip()

results_overview_text = """
The final evaluation combined repeated cross-validation, negative-set sensitivity analysis, fusion strategy comparison, Top-K ranking utility, biological validation and leakage/bias audits. The most robust pattern was that protein-containing models substantially outperformed the genomic-only baseline. Multimodal models provided modest but reproducible improvements in ranking-oriented metrics, especially PR-AUC and Top-K recovery. DNABERT-2 multimodal models were competitive with handcrafted multimodal models but should be interpreted as a ranking extension rather than a clearly superior replacement. Top-K ranking utility provided the strongest practical evidence for gene prioritization: multimodal rankings recovered many more current-positive genes in the top-ranked lists than expected by random sampling.
""".strip()

discussion_text = """
The results support a cautious interpretation of the proposed framework. The strongest conclusion is not that the model definitively classifies T2D and non-T2D genes, but that sequence-derived protein and genomic representations can support useful gene prioritization. Protein embeddings provided the dominant signal, while genomic features contributed complementary but modest information. DNABERT-2 changed ranking behaviour and provided directional improvements in selected analyses, but its advantage over handcrafted genomic features was not statistically established across all metrics. External and pathway-level validation were supportive rather than confirmatory, consistent with incomplete disease-gene labels and the limitations of binary negative sampling. Leakage and bias audits did not identify severe duplicate leakage, and gene-symbol-length matching did not reduce model performance; nevertheless, nuisance-only signal and cluster-aware performance reduction indicate residual dataset-level bias that should be acknowledged.
""".strip()

limitations_text = """
Several limitations should be emphasized. First, the task is gene/protein-level prioritization rather than patient-level T2D diagnosis. Second, negative labels represent genes without current T2D evidence, not confirmed non-disease genes. Third, external and pathway validation did not remain significant after correction and should be treated as hypothesis-generating. Fourth, protein embedding cluster-aware evaluation showed modest performance reduction, suggesting that part of the signal may reflect embedding-neighborhood or protein-family similarity. Fifth, nuisance-only models achieved moderate performance, indicating residual dataset-level bias. These limitations motivate future work using independent external validation sets, more stringent homology-aware splits, improved negative sampling, tissue-specific regulatory data and experimental validation of prioritized candidates.
""".strip()

conclusion_text = """
Overall, the study provides a robust and explainable sequence-based framework for T2D gene prioritization. The framework demonstrates strong Top-K prioritization utility, modest multimodal gains, and interpretable modality-level behaviour. The results support the value of combining protein and genomic sequence representations while emphasizing that biological findings remain hypothesis-generating under incomplete disease-gene labels.
""".strip()

wording_df = pd.DataFrame([
    {"section": "Abstract Results", "text": abstract_results_text},
    {"section": "Results Overview", "text": results_overview_text},
    {"section": "Discussion", "text": discussion_text},
    {"section": "Limitations", "text": limitations_text},
    {"section": "Conclusion", "text": conclusion_text},
])

display(wording_df)

save_df(
    wording_df,
    RESULT_DIR / "phase25_final_manuscript_ready_wording.csv"
)

with open(REPORT_DIR / "phase25_final_manuscript_ready_wording.md", "w") as f:
    for _, row in wording_df.iterrows():
        write_markdown_section(f, row["section"], row["text"])

print("Saved final wording.")

,section,text
0,Abstract Results,"Across repeated stratified cross-validation, protein-containing models consistently outperformed the genomic-only baseline. Multimodal models improved ranking-oriented metrics relative to protein-only models, although gains were modest. DNABERT-2 multimodal representations achieved directional improvements in some settings but did not establish clear superiority over handcrafted genomic features after multiple-testing correction. Top-K ranking analysis demonstrated strong gene-prioritization..."
1,Results Overview,"The final evaluation combined repeated cross-validation, negative-set sensitivity analysis, fusion strategy comparison, Top-K ranking utility, biological validation and leakage/bias audits. The most robust pattern was that protein-containing models substantially outperformed the genomic-only baseline. Multimodal models provided modest but reproducible improvements in ranking-oriented metrics, especially PR-AUC and Top-K recovery. DNABERT-2 multimodal models were competitive with handcrafted ..."
2,Discussion,"The results support a cautious interpretation of the proposed framework. The strongest conclusion is not that the model definitively classifies T2D and non-T2D genes, but that sequence-derived protein and genomic representations can support useful gene prioritization. Protein embeddings provided the dominant signal, while genomic features contributed complementary but modest information. DNABERT-2 changed ranking behaviour and provided directional improvements in selected analyses, but its a..."
3,Limitations,"Several limitations should be emphasized. First, the task is gene/protein-level prioritization rather than patient-level T2D diagnosis. Second, negative labels represent genes without current T2D evidence, not confirmed non-disease genes. Third, external and pathway validation did not remain significant after correction and should be treated as hypothesis-generating. Fourth, protein embedding cluster-aware evaluation showed modest performance reduction, suggesting that part of the signal may..."
4,Conclusion,"Overall, the study provides a robust and explainable sequence-based framework for T2D gene prioritization. The framework demonstrates strong Top-K prioritization utility, modest multimodal gains, and interpretable modality-level behaviour. The results support the value of combining protein and genomic sequence representations while emphasizing that biological findings remain hypothesis-generating under incomplete disease-gene labels."


Saved: /content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/results/phase25_final_manuscript_ready_wording.csv
Saved final wording.


In [8]:
# ============================================================
# RECOMMENDED MAIN RESULT TABLES
# ============================================================

main_table_records = [
    {
        "table_id": "Table 1",
        "content": "Dataset and feature representation summary",
        "source": "Existing manuscript / Phase 3 / Phase 16",
        "main_or_supplement": "Main",
        "notes": "Include n=1,806; 903 positive / 903 background; feature dimensions."
    },
    {
        "table_id": "Table 2",
        "content": "Repeated CV performance summary",
        "source": "Phase 17",
        "main_or_supplement": "Main",
        "notes": "PR-AUC, ROC-AUC, MCC, F1 mean ± SD."
    },
    {
        "table_id": "Table 3",
        "content": "Top-K ranking utility",
        "source": "Phase 24",
        "main_or_supplement": "Main",
        "notes": "Top-50, Top-100, Top-200 Precision@K, N positives, enrichment."
    },
    {
        "table_id": "Table 4",
        "content": "Claim strength and validation summary",
        "source": "Phase 25",
        "main_or_supplement": "Main or Supplementary",
        "notes": "Useful for transparent paper narrative."
    },
    {
        "table_id": "Supplementary Table S1",
        "content": "Negative-set sensitivity",
        "source": "Phase 18",
        "main_or_supplement": "Supplementary",
        "notes": "All scenarios."
    },
    {
        "table_id": "Supplementary Table S2",
        "content": "External/pathway validation",
        "source": "Phase 19 / 20",
        "main_or_supplement": "Supplementary",
        "notes": "Supportive/hypothesis-generating only."
    },
    {
        "table_id": "Supplementary Table S3",
        "content": "Case-level gene audit",
        "source": "Phase 21",
        "main_or_supplement": "Supplementary",
        "notes": "Include NDUFV1, NDUFA12, GIPR, GCKR, NKX6-1, etc."
    },
    {
        "table_id": "Supplementary Table S4",
        "content": "Leakage and bias audit",
        "source": "Phase 23 / 23B",
        "main_or_supplement": "Supplementary",
        "notes": "Duplicate, nearest-neighbor, nuisance-only, matched sampling."
    }
]

main_table_recommendation_df = pd.DataFrame(main_table_records)
display(main_table_recommendation_df)

save_df(
    main_table_recommendation_df,
    RESULT_DIR / "phase25_table_recommendation.csv"
)

,table_id,content,source,main_or_supplement,notes
0,Table 1,Dataset and feature representation summary,Existing manuscript / Phase 3 / Phase 16,Main,"Include n=1,806; 903 positive / 903 background; feature dimensions."
1,Table 2,Repeated CV performance summary,Phase 17,Main,"PR-AUC, ROC-AUC, MCC, F1 mean ± SD."
2,Table 3,Top-K ranking utility,Phase 24,Main,"Top-50, Top-100, Top-200 Precision@K, N positives, enrichment."
3,Table 4,Claim strength and validation summary,Phase 25,Main or Supplementary,Useful for transparent paper narrative.
4,Supplementary Table S1,Negative-set sensitivity,Phase 18,Supplementary,All scenarios.
5,Supplementary Table S2,External/pathway validation,Phase 19 / 20,Supplementary,Supportive/hypothesis-generating only.
6,Supplementary Table S3,Case-level gene audit,Phase 21,Supplementary,"Include NDUFV1, NDUFA12, GIPR, GCKR, NKX6-1, etc."
7,Supplementary Table S4,Leakage and bias audit,Phase 23 / 23B,Supplementary,"Duplicate, nearest-neighbor, nuisance-only, matched sampling."


Saved: /content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/results/phase25_table_recommendation.csv


In [9]:
# ============================================================
# EXPORT EXCEL WORKBOOK
# ============================================================

excel_path = EXCEL_DIR / "phase25_final_manuscript_results_integration.xlsx"

tables = {
    "Final_Performance_Summary": final_performance_summary_df,
    "Claim_Strength_Audit": claim_strength_df,
    "Figure_Recommendations": figure_recommendation_df,
    "Table_Recommendations": main_table_recommendation_df,
    "Final_Wording": wording_df,
    "Phase17_CV": phase17_cv,
    "Phase17_Paired": phase17_paired,
    "Phase22_Fusion_Summary": phase22_summary,
    "Phase22_Paired": phase22_paired,
    "Phase24_TopK": phase24_topk,
    "Phase24_Random": phase24_random,
    "Phase24_Bio": phase24_bio,
    "Phase20_Pathway": phase20_pathway,
    "Phase20_Network": phase20_network,
    "Phase21_Cases": phase21_cases,
    "Phase23B_Matched": phase23b_matched,
}

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, df in tables.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print("Saved Excel:", excel_path)

Saved Excel: /content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/excel/phase25_final_manuscript_results_integration.xlsx


In [10]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("=== PHASE 25 COMPLETE ===")

print("\nFinal performance summary:")
display(final_performance_summary_df)

print("\nClaim strength audit:")
display(claim_strength_df)

print("\nFigure recommendations:")
display(figure_recommendation_df)

print("\nTable recommendations:")
display(main_table_recommendation_df)

print("\nFinal manuscript wording:")
display(wording_df)

print("\nResults:")
for p in sorted(RESULT_DIR.glob("*.csv")):
    print(p)

print("\nExcel:")
for p in sorted(EXCEL_DIR.glob("*.xlsx")):
    print(p)

print("\nReports:")
for p in sorted(REPORT_DIR.glob("*")):
    print(p)

=== PHASE 25 COMPLETE ===

Final performance summary:


,evidence_layer,model_or_analysis,main_result,interpretation
0,Repeated CV benchmark,DNABERT-2 multimodal,PR-AUC 0.716 ± 0.024; MCC 0.331 ± 0.052; ROC-AUC 0.731 ± 0.023,Core internal robustness benchmark.
1,Repeated CV benchmark,Handcrafted multimodal,PR-AUC 0.714 ± 0.019; MCC 0.317 ± 0.049; ROC-AUC 0.725 ± 0.020,Core internal robustness benchmark.
2,Repeated CV benchmark,Protein-only,PR-AUC 0.695 ± 0.024; MCC 0.308 ± 0.033; ROC-AUC 0.713 ± 0.020,Core internal robustness benchmark.
3,Repeated CV benchmark,Genomic-only,PR-AUC 0.645 ± 0.029; MCC 0.203 ± 0.043; ROC-AUC 0.641 ± 0.029,Core internal robustness benchmark.
4,Fusion strategy comparison,Best PR-AUC fusion,Stacking_Protein_Handcrafted_DNABERT2 achieved PR-AUC 0.721 ± 0.020,Decision-level fusion provides small ranking gains.
5,Fusion strategy comparison,Best MCC fusion,Early_fusion_Protein_DNABERT2Genomic achieved MCC 0.339 ± 0.054,Early DNABERT-2 fusion remains strong for threshold-based classification.
6,Top-K ranking utility,Handcrafted multimodal,Top-100 recovered 84/100 positives; Precision@100=0.840,Strong gene-prioritization utility.



Claim strength audit:


,claim,support_level,evidence,safe_wording
0,Protein-containing models outperform genomic-only models.,Strong,"Repeated CV, negative sensitivity, protein-only vs genomic-only paired results.",Protein-containing models consistently outperformed the genomic-only baseline.
1,Multimodal models improve ranking over protein-only.,Moderate to strong,Repeated CV PR-AUC improvements; Top-K utility; Phase 22 fusion comparison.,"Multimodal models improved ranking-oriented metrics relative to protein-only models, although gains were modest."
2,DNABERT-2 is superior to handcrafted multimodal.,Weak / not established,"DNABERT-2 had small directional gains in repeated CV and some fusion settings, but not consistently significant after correction.",DNABERT-2 provided a directional ranking extension but did not establish clear superiority over handcrafted genomic features.
3,Top-K gene prioritization utility is strong.,Strong,Top-K current-positive recovery substantially exceeded random baselines across K thresholds.,"Top-K analysis demonstrated strong prioritization utility, with multimodal rankings recovering substantially more current-positive genes than random same-size gene sets."
4,External/pathway biological validation is confirmatory.,Weak,External/pathway enrichment did not remain significant after FDR correction in Phases 19–20.,Biological validation was supportive and hypothesis-generating rather than confirmatory.
5,Mitochondrial/OXPHOS module is biologically meaningful.,Supportive / hypothesis-generating,"Case-level and network analyses highlighted NDUF/OXPHOS candidates, but FDR-corrected enrichment was not significant.",Mitochondrial/OXPHOS candidates emerged as a hypothesis-generating module-level signal.
6,The study is free from leakage and bias.,Not acceptable,"No serious duplicate leakage, but nuisance-only signal and cluster-aware performance drop exist.","Leakage and bias audits did not reveal severe duplication leakage, but residual dataset-level bias and embedding-neighborhood effects were detected and explicitly controlled."
7,Bias controls preserve main conclusions.,Moderate to strong,Gene-symbol-length matching did not reduce main model performance.,Bias-controlled matching showed that gene-symbol-length imbalance did not explain the main sequence-based ranking results.



Figure recommendations:


...


Table recommendations:


,table_id,content,source,main_or_supplement,notes
0,Table 1,Dataset and feature representation summary,Existing manuscript / Phase 3 / Phase 16,Main,"Include n=1,806; 903 positive / 903 background; feature dimensions."
1,Table 2,Repeated CV performance summary,Phase 17,Main,"PR-AUC, ROC-AUC, MCC, F1 mean ± SD."
2,Table 3,Top-K ranking utility,Phase 24,Main,"Top-50, Top-100, Top-200 Precision@K, N positives, enrichment."
3,Table 4,Claim strength and validation summary,Phase 25,Main or Supplementary,Useful for transparent paper narrative.
4,Supplementary Table S1,Negative-set sensitivity,Phase 18,Supplementary,All scenarios.
5,Supplementary Table S2,External/pathway validation,Phase 19 / 20,Supplementary,Supportive/hypothesis-generating only.
6,Supplementary Table S3,Case-level gene audit,Phase 21,Supplementary,"Include NDUFV1, NDUFA12, GIPR, GCKR, NKX6-1, etc."
7,Supplementary Table S4,Leakage and bias audit,Phase 23 / 23B,Supplementary,"Duplicate, nearest-neighbor, nuisance-only, matched sampling."



Final manuscript wording:


,section,text
0,Abstract Results,"Across repeated stratified cross-validation, protein-containing models consistently outperformed the genomic-only baseline. Multimodal models improved ranking-oriented metrics relative to protein-only models, although gains were modest. DNABERT-2 multimodal representations achieved directional improvements in some settings but did not establish clear superiority over handcrafted genomic features after multiple-testing correction. Top-K ranking analysis demonstrated strong gene-prioritization..."
1,Results Overview,"The final evaluation combined repeated cross-validation, negative-set sensitivity analysis, fusion strategy comparison, Top-K ranking utility, biological validation and leakage/bias audits. The most robust pattern was that protein-containing models substantially outperformed the genomic-only baseline. Multimodal models provided modest but reproducible improvements in ranking-oriented metrics, especially PR-AUC and Top-K recovery. DNABERT-2 multimodal models were competitive with handcrafted ..."
2,Discussion,"The results support a cautious interpretation of the proposed framework. The strongest conclusion is not that the model definitively classifies T2D and non-T2D genes, but that sequence-derived protein and genomic representations can support useful gene prioritization. Protein embeddings provided the dominant signal, while genomic features contributed complementary but modest information. DNABERT-2 changed ranking behaviour and provided directional improvements in selected analyses, but its a..."
3,Limitations,"Several limitations should be emphasized. First, the task is gene/protein-level prioritization rather than patient-level T2D diagnosis. Second, negative labels represent genes without current T2D evidence, not confirmed non-disease genes. Third, external and pathway validation did not remain significant after correction and should be treated as hypothesis-generating. Fourth, protein embedding cluster-aware evaluation showed modest performance reduction, suggesting that part of the signal may..."
4,Conclusion,"Overall, the study provides a robust and explainable sequence-based framework for T2D gene prioritization. The framework demonstrates strong Top-K prioritization utility, modest multimodal gains, and interpretable modality-level behaviour. The results support the value of combining protein and genomic sequence representations while emphasizing that biological findings remain hypothesis-generating under incomplete disease-gene labels."



Results:
/content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/results/phase25_claim_strength_audit.csv
/content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/results/phase25_figure_recommendation_table.csv
/content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/results/phase25_final_manuscript_ready_wording.csv
/content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/results/phase25_final_performance_summary.csv
/content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/results/phase25_table_recommendation.csv

Excel:
/content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/excel/phase25_final_manuscript_results_integration.xlsx

Reports:
/content/drive/MyDrive/Project_Protein/model/phase25_final_manuscript_results_integration/reports/phase25_final_manuscript_ready_wording.md


In [11]:
display(final_performance_summary_df)
display(claim_strength_df)
display(figure_recommendation_df)
display(main_table_recommendation_df)
display(wording_df)

,evidence_layer,model_or_analysis,main_result,interpretation
0,Repeated CV benchmark,DNABERT-2 multimodal,PR-AUC 0.716 ± 0.024; MCC 0.331 ± 0.052; ROC-AUC 0.731 ± 0.023,Core internal robustness benchmark.
1,Repeated CV benchmark,Handcrafted multimodal,PR-AUC 0.714 ± 0.019; MCC 0.317 ± 0.049; ROC-AUC 0.725 ± 0.020,Core internal robustness benchmark.
2,Repeated CV benchmark,Protein-only,PR-AUC 0.695 ± 0.024; MCC 0.308 ± 0.033; ROC-AUC 0.713 ± 0.020,Core internal robustness benchmark.
3,Repeated CV benchmark,Genomic-only,PR-AUC 0.645 ± 0.029; MCC 0.203 ± 0.043; ROC-AUC 0.641 ± 0.029,Core internal robustness benchmark.
4,Fusion strategy comparison,Best PR-AUC fusion,Stacking_Protein_Handcrafted_DNABERT2 achieved PR-AUC 0.721 ± 0.020,Decision-level fusion provides small ranking gains.
5,Fusion strategy comparison,Best MCC fusion,Early_fusion_Protein_DNABERT2Genomic achieved MCC 0.339 ± 0.054,Early DNABERT-2 fusion remains strong for threshold-based classification.
6,Top-K ranking utility,Handcrafted multimodal,Top-100 recovered 84/100 positives; Precision@100=0.840,Strong gene-prioritization utility.


,claim,support_level,evidence,safe_wording
0,Protein-containing models outperform genomic-only models.,Strong,"Repeated CV, negative sensitivity, protein-only vs genomic-only paired results.",Protein-containing models consistently outperformed the genomic-only baseline.
1,Multimodal models improve ranking over protein-only.,Moderate to strong,Repeated CV PR-AUC improvements; Top-K utility; Phase 22 fusion comparison.,"Multimodal models improved ranking-oriented metrics relative to protein-only models, although gains were modest."
2,DNABERT-2 is superior to handcrafted multimodal.,Weak / not established,"DNABERT-2 had small directional gains in repeated CV and some fusion settings, but not consistently significant after correction.",DNABERT-2 provided a directional ranking extension but did not establish clear superiority over handcrafted genomic features.
3,Top-K gene prioritization utility is strong.,Strong,Top-K current-positive recovery substantially exceeded random baselines across K thresholds.,"Top-K analysis demonstrated strong prioritization utility, with multimodal rankings recovering substantially more current-positive genes than random same-size gene sets."
4,External/pathway biological validation is confirmatory.,Weak,External/pathway enrichment did not remain significant after FDR correction in Phases 19–20.,Biological validation was supportive and hypothesis-generating rather than confirmatory.
5,Mitochondrial/OXPHOS module is biologically meaningful.,Supportive / hypothesis-generating,"Case-level and network analyses highlighted NDUF/OXPHOS candidates, but FDR-corrected enrichment was not significant.",Mitochondrial/OXPHOS candidates emerged as a hypothesis-generating module-level signal.
6,The study is free from leakage and bias.,Not acceptable,"No serious duplicate leakage, but nuisance-only signal and cluster-aware performance drop exist.","Leakage and bias audits did not reveal severe duplication leakage, but residual dataset-level bias and embedding-neighborhood effects were detected and explicitly controlled."
7,Bias controls preserve main conclusions.,Moderate to strong,Gene-symbol-length matching did not reduce main model performance.,Bias-controlled matching showed that gene-symbol-length imbalance did not explain the main sequence-based ranking results.


,figure_id,recommended_content,source_phase,main_or_supplement,reason
0,Figure 1,Overall workflow diagram,Phase 16,Main,"Shows dataset construction, feature extraction, evaluation, XAI and validation pipeline."
1,Figure 2,Repeated CV model performance,Phase 17,Main,Core benchmark result.
2,Figure 3,Top-K Precision/Enrichment and observed vs random Top-100 recovery,Phase 24,Main,Best evidence for gene-prioritization utility.
3,Figure 4,Fusion strategy comparison,Phase 22,Main or Supplementary,Supports multimodal integration analysis.
4,Figure 5,Block-level modality importance,Phase 14 / 16,Main,Supports explainable modality contribution.
5,Figure 6,Case-level candidate/pathway-supported genes,Phase 21,Main or Supplementary,Strengthens explainability under incomplete labels.
6,Supplementary Figure S1,Negative-set sensitivity plots,Phase 18,Supplementary,Robustness evidence.
7,Supplementary Figure S2,Pathway/network validation plots,Phase 20,Supplementary,Supportive but not confirmatory biological evidence.
8,Supplementary Figure S3,Leakage/bias audit plots,Phase 23 / 23B,Supplementary,Addresses reviewer concerns without overcrowding main text.


,table_id,content,source,main_or_supplement,notes
0,Table 1,Dataset and feature representation summary,Existing manuscript / Phase 3 / Phase 16,Main,"Include n=1,806; 903 positive / 903 background; feature dimensions."
1,Table 2,Repeated CV performance summary,Phase 17,Main,"PR-AUC, ROC-AUC, MCC, F1 mean ± SD."
2,Table 3,Top-K ranking utility,Phase 24,Main,"Top-50, Top-100, Top-200 Precision@K, N positives, enrichment."
3,Table 4,Claim strength and validation summary,Phase 25,Main or Supplementary,Useful for transparent paper narrative.
4,Supplementary Table S1,Negative-set sensitivity,Phase 18,Supplementary,All scenarios.
5,Supplementary Table S2,External/pathway validation,Phase 19 / 20,Supplementary,Supportive/hypothesis-generating only.
6,Supplementary Table S3,Case-level gene audit,Phase 21,Supplementary,"Include NDUFV1, NDUFA12, GIPR, GCKR, NKX6-1, etc."
7,Supplementary Table S4,Leakage and bias audit,Phase 23 / 23B,Supplementary,"Duplicate, nearest-neighbor, nuisance-only, matched sampling."


,section,text
0,Abstract Results,"Across repeated stratified cross-validation, protein-containing models consistently outperformed the genomic-only baseline. Multimodal models improved ranking-oriented metrics relative to protein-only models, although gains were modest. DNABERT-2 multimodal representations achieved directional improvements in some settings but did not establish clear superiority over handcrafted genomic features after multiple-testing correction. Top-K ranking analysis demonstrated strong gene-prioritization..."
1,Results Overview,"The final evaluation combined repeated cross-validation, negative-set sensitivity analysis, fusion strategy comparison, Top-K ranking utility, biological validation and leakage/bias audits. The most robust pattern was that protein-containing models substantially outperformed the genomic-only baseline. Multimodal models provided modest but reproducible improvements in ranking-oriented metrics, especially PR-AUC and Top-K recovery. DNABERT-2 multimodal models were competitive with handcrafted ..."
2,Discussion,"The results support a cautious interpretation of the proposed framework. The strongest conclusion is not that the model definitively classifies T2D and non-T2D genes, but that sequence-derived protein and genomic representations can support useful gene prioritization. Protein embeddings provided the dominant signal, while genomic features contributed complementary but modest information. DNABERT-2 changed ranking behaviour and provided directional improvements in selected analyses, but its a..."
3,Limitations,"Several limitations should be emphasized. First, the task is gene/protein-level prioritization rather than patient-level T2D diagnosis. Second, negative labels represent genes without current T2D evidence, not confirmed non-disease genes. Third, external and pathway validation did not remain significant after correction and should be treated as hypothesis-generating. Fourth, protein embedding cluster-aware evaluation showed modest performance reduction, suggesting that part of the signal may..."
4,Conclusion,"Overall, the study provides a robust and explainable sequence-based framework for T2D gene prioritization. The framework demonstrates strong Top-K prioritization utility, modest multimodal gains, and interpretable modality-level behaviour. The results support the value of combining protein and genomic sequence representations while emphasizing that biological findings remain hypothesis-generating under incomplete disease-gene labels."
